# Act 6: compose an assistant with a governed workflow

**Agent systems:** the outer assistant selects a confirmed request and calls Act 3 as a tool. The inner cart proposer recommends items; the workflow validates, requests host approval when needed, and performs mock checkout. Skills help explain the accepted menu.

Start with repeatable **offline fixtures**. This notebook requires Python 3.13+; Colab runtime compatibility has not been validated here. Orders and stock changes exist only in memory. No real purchase or email occurs.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

root = Path.cwd()
if not (root / "formaggio" / "config.py").exists():
    if (root.parent / "formaggio" / "config.py").exists():
        root = root.parent
    else:
        root = root / "agent-harness"
        if not root.exists():
            subprocess.run(["git", "clone", "https://github.com/dlops-io/agent-harness.git", str(root)], check=True)
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Repository:", root)

In [ ]:
import tomllib

if sys.version_info < (3, 13):
    raise RuntimeError("Select a Python 3.13+ notebook kernel before continuing.")
project = tomllib.loads(Path("pyproject.toml").read_text())
subprocess.run([sys.executable, "-m", "pip", "install", *project["project"]["dependencies"]], check=True)
subprocess.run([sys.executable, "-m", "scripts.check_sdk"], check=True)

## 1. Run Act 6

The builder describes the agent and its named layers. Calling `.run(...)` starts a fresh session, checkout, workflow handles, counters, and skill-loading state.


In [ ]:
from pathlib import Path
from acts.act6_composition import build_act6, build_assistant, build_order_tools
from formaggio.operations.observability import Recorder
from cli import print_composition_result

DB_PATH = Path("outputs/act6_notebook.sqlite")


In [ ]:
lesson = build_act6(fixture=True)
with Recorder(DB_PATH) as recorder:
    result = await lesson.run(recorder, progress=print)
print_composition_result(result)


## 2. Read the agent and tool boundary

**Context engineering:** the host supplies confirmed requests and refreshes authoritative order state before each model call. The model supplies only `request_id` to a workflow tool. It cannot rewrite allergies or pass an approval flag.

**Agent harness:** tracing, budgets, planning, compaction, and skills support execution. The existing workflow owns mandatory validation, exact-ticket approval, and final inventory checks.


In [ ]:
import inspect
print(inspect.getsource(build_order_tools))
print(inspect.getsource(build_assistant))
print(inspect.getsource(build_act6))


## 3. Keep the host in control of approval

The workflow can return `pending_approval`. The assistant yields; the host callback decides; the same workflow resumes. The callback below is an explicitly scripted decline, not a model decision. Manager deliberation is outside the shared active-time budget.


In [ ]:
async def fixture_decline(ticket):
    print("Manager ticket:", ticket.ticket_id)
    return "decline"

approval_lesson = build_act6(fixture=True, scenario="manager-approval", decision_source="fixture")
with Recorder(DB_PATH) as recorder:
    declined_result = await approval_lesson.run(recorder, manager=fixture_decline, progress=print)
print_composition_result(declined_result)


## 4. Remove optional layers; keep workflow checks

Remove each named feature to see what belongs to the harness. The task and skill tools disappear; detailed tracing and history eviction stop. Current request and order state still reach the model, and the ordering workflow still enforces its checks. Removing `budget` removes the explicit call and time allowances; the SDK's fallback loop bound remains.


In [ ]:
minimal = approval_lesson
for name in ("trace", "budget", "planning", "compaction", "skills"):
    minimal = minimal.without(name)
with Recorder(DB_PATH) as recorder:
    minimal_result = await minimal.run(recorder, manager=fixture_decline, progress=print)
print_composition_result(minimal_result)
assert minimal_result["orders"][0]["status"] == "declined"
assert not minimal_result["order_placed"]


## 5. Two approved requests compete for shared stock

Within one invocation both workflows share the same checkout. Both pass initial validation, but the first approved order consumes inventory. Final validation blocks the second order even though its manager also approved it.

Separate `.run(...)` calls get fresh checkouts by default. A host may explicitly pass `checkout=...` to share inventory across invocations.


In [ ]:
async def fixture_approve(ticket):
    print("Approve fixture ticket:", ticket.ticket_id)
    return "approve"

two_order_lesson = build_act6(fixture=True, scenario="two-orders", decision_source="fixture")
with Recorder(DB_PATH) as recorder:
    competing_result = await two_order_lesson.run(recorder, manager=fixture_approve, progress=print)
print_composition_result(competing_result)
assert [order["status"] for order in competing_result["orders"]] == ["placed", "blocked"]


## 6. Inspect evidence and try live decisions

**Evaluation:** inspect actual receipts, policy events, loaded skills, and model inputs. The fixture prescribes tool choices and has zero workflow model calls. Live mode uses a model for both the outer assistant and inner proposals; their counts and limits are separate.

The default budget allows 16 outer calls and 20 outer tool calls, plus 8 proposer calls per workflow. A shared 180-second active timer covers nested work and host resumes, with a 120-second cap on each proposal. Review waits do not reset those allowances.


In [ ]:
with Recorder(DB_PATH) as recorder:
    events = recorder.timeline(competing_result["run_id"])
    states = [event["payload"] for event in events if event["event_type"] == "composition.order_state"]
for state in states:
    print(state["request_id"], state["status"], "order placed:", state["order_placed"])
print("Loaded skills:", competing_result["skills_loaded"])
print("Outer calls:", competing_result["model_calls"], "Workflow calls:", competing_result["workflow_model_calls"])


In [ ]:
import os
from getpass import getpass
from formaggio.config import MODEL

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

async def review_order(ticket):
    print(ticket.model_dump_json(indent=2))
    while True:
        decision = input("Approve mock order or decline? ").strip().lower()
        if decision in {"approve", "decline"}:
            return decision

with Recorder(DB_PATH) as recorder:
    live_result = await build_act6(model=MODEL, scenario="manager-approval").run(
        recorder, manager=review_order, progress=print)
print_composition_result(live_result)


## Reflect

- Which decisions belong to the outer assistant, the cart proposer, and the workflow?
- How does the current-state capsule connect context engineering to execution?
- Why does removing skills still allow governed checkout?
- Why do two approved tickets produce only one receipt in the competing-order example?
- Which recorded evidence distinguishes a proposed cart, a pending approval, and a placed mock order?
